# Data Science e IA dentro de Docker
Natalia Hernández

Se cargan datos Iris incluidos en scikit-learn, se separan entrenamiento/prueba y se realiza preprocesamiento sin fuga de datos. Se comparan una regresión logística y una red neuronal TensorFlow. Ejecución en CPU.


In [ ]:
import sys, platform, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import sklearn
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
np.random.seed(42)
tf.keras.utils.set_random_seed(42)
print(json.dumps({'python': sys.version.split()[0], 'arquitectura': platform.machine(), 'tensorflow': tf.__version__, 'scikit-learn': sklearn.__version__, 'GPU': len(tf.config.list_physical_devices('GPU'))}, indent=2))


In [ ]:
iris = load_iris(as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(iris.data, iris.target, test_size=0.2, random_state=42, stratify=iris.target)
print(f'Entrenamiento: {len(X_train)} / Prueba: {len(X_test)}')
iris.frame.head()


## Preprocesamiento y scikit-learn
El escalador aprende solamente del conjunto de entrenamiento mediante un Pipeline.


In [ ]:
pipeline = make_pipeline(StandardScaler(), LogisticRegression(max_iter=500, random_state=42))
pipeline.fit(X_train, y_train)
pred = pipeline.predict(X_test)
acc_sklearn = accuracy_score(y_test, pred)
print(f'Exactitud scikit-learn: {acc_sklearn:.4f}')
print(classification_report(y_test, pred, target_names=iris.target_names))
assert acc_sklearn >= 0.8


## Red neuronal TensorFlow
Se entrena en CPU con lotes pequeños y un orden reproducible. El conjunto de prueba se usa al final.


In [ ]:
scaler = StandardScaler()
train = scaler.fit_transform(X_train).astype('float32')
test = scaler.transform(X_test).astype('float32')
model = tf.keras.Sequential([tf.keras.layers.Input(shape=(4,)), tf.keras.layers.Dense(16, activation='relu'), tf.keras.layers.Dense(3, activation='softmax')])
model.compile(optimizer=tf.keras.optimizers.Adam(0.01), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
for epoch in range(120):
    model.train_on_batch(train, y_train.to_numpy())
pred_tf = np.argmax(model(test, training=False).numpy(), axis=1)
acc_tf = accuracy_score(y_test, pred_tf)
print(f'Exactitud TensorFlow: {acc_tf:.4f}')
assert acc_tf >= 0.8


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=iris.target_names, ax=axes[0], colorbar=False)
axes[0].set_title('scikit-learn')
ConfusionMatrixDisplay.from_predictions(y_test, pred_tf, display_labels=iris.target_names, ax=axes[1], colorbar=False)
axes[1].set_title('TensorFlow')
plt.tight_layout()
plt.show()


In [ ]:
out = Path('resultados')
out.mkdir(exist_ok=True)
metrics = {'muestras_entrenamiento': len(X_train), 'muestras_prueba': len(X_test), 'exactitud_sklearn': float(acc_sklearn), 'exactitud_tensorflow': float(acc_tf), 'tensorflow': tf.__version__, 'scikit_learn': sklearn.__version__, 'dispositivo': 'CPU'}
(out / 'metricas.json').write_text(json.dumps(metrics, indent=2))
model.save(out / 'iris.keras')
print(json.dumps(metrics, indent=2))
print('Modelo y métricas guardados en el directorio compartido.')


## Interpretación
Las métricas corresponden a 30 muestras de prueba del conjunto Iris y demuestran el funcionamiento del entorno. No sustituyen una evaluación con datos independientes. El directorio montado conserva el notebook, las métricas y el modelo al recrear el contenedor.
